In [2]:
import numpy as np
import glob
import lgdo.lh5 as lh5
import os, json
import copy
import glob
import matplotlib.pyplot as plt
from pygama.pargen.utils import load_data
from legendmeta import LegendMetadata
from dbetto import Props, TextDB, AttrsDict
import pandas as pd
from tqdm.notebook import tqdm
import awkward as ak
from resolution_extraction import get_eres_per_detector, get_expo_per_detector


from helper_lib import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [3]:
version = "v2.1.5"
base = f"/global/cfs/projectdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/{version}/"
scratch_folder = "/pscratch/sd/b/borrfran/sim-v1.1.0-20260401/"
# metadata2_path = os.environ["METADATA"]
metadata2_path = "/global/homes/b/borrfran/workspace/l200/legend-metadata"
config = Props.read_from(base+"/config.json", subst_pathvar=True)['setups']['l200']['paths']


meta = LegendMetadata(config['metadata'])
meta2 = LegendMetadata(metadata2_path)

timestamp = meta.dataprod.runinfo.p03.r000.phy.start_key

chmap = meta.channelmap(timestamp)

DET_TYPES = ("BEGe", "COAX", "ICPC", "PPC")

simulated_energies = range(200, 1100, 100)

could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f3e2654cfe0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f3e2654d300>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f3e2654eca0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f3e2654df30>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f3e265737e0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f3e265710d0>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldat

In [4]:
periods_simulated = ['p03', 'p04', 'p06', 'p07', 'p08', 'p09']
periods_metadata = meta2.datasets.runlists['valid']['phy']

periods_dict = {
    k: [item for rng in v for item in expand_range(rng)]
    for k, v in periods_metadata.items()
}

keys = list(periods_dict.keys())
for key in keys:
    if key not in periods_simulated:
        periods_dict.pop(key)

In [5]:
periods_dict

{'p03': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005'],
 'p04': ['r000', 'r001', 'r002', 'r003'],
 'p06': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005'],
 'p07': ['r002', 'r003', 'r004', 'r005', 'r006', 'r007'],
 'p08': ['r000',
  'r001',
  'r002',
  'r003',
  'r004',
  'r006',
  'r007',
  'r008',
  'r009',
  'r010',
  'r011',
  'r012',
  'r013',
  'r014'],
 'p09': ['r000', 'r001', 'r002', 'r003', 'r004', 'r005']}

## Populate 'expo_dict'

In [33]:
expo_dict = get_expo_per_detector(meta2, periods_dict)
Props.write_to("./v1/dictionaries/expo_dict_withp09-r004.yaml", expo_dict)

p09-r005 (expo): 100%|██████████| 101/101 [00:00<00:00, 4782.13it/s]


## Read 'expo_dict' from file

In [6]:
DET_TYPE_MAP = {"B": "BEGe", "C": "COAX", "V": "ICPC", "P": "PPC"}

expo_dict = Props.read_from("./v1/dictionaries/expo_dict_withp09-r004.yaml")

In [7]:
exposures = {
    'ICPC': [],
    'BEGe': [],
    'PPC': [],
    'COAX': []
}

for p in periods_dict:
    for r in periods_dict[p]:

        for key, values in expo_dict[p][r].items():

            if expo_dict[p][r][key]['usability'] != 'on': continue
            exposures[DET_TYPE_MAP[key[0]]].append(expo_dict[p][r][key]['expo'])
        

In [37]:
exposures_sum = {}
for key in exposures:
    exposures_sum[key] = np.sum(exposures[key])

In [38]:
exposures_sum

{'ICPC': 45.89325228610541,
 'BEGe': 10.911190302177605,
 'PPC': 9.661264611630795,
 'COAX': 7.759765955585976}

## Check on V04549A

In [17]:
tot_expo_det = 0
counts = 0
det = "C000RG1"

for period, run_dict in expo_dict.items():

    for run, det_dict in run_dict.items():
        print(det, period, run, det_dict[det]['usability'], det_dict[det]['expo'])
        if det_dict[det]['usability'] != 'on':
            print(det, period, run, det_dict[det]['usability'])
            continue
        tot_expo_det+=det_dict[det]['expo']
        counts +=1

tot_expo_det, counts

C000RG1 p03 r000 on 0.029711467918979897
C000RG1 p03 r001 on 0.03687719851953253
C000RG1 p03 r002 on 0.03713833054478161
C000RG1 p03 r003 on 0.02988555593581261
C000RG1 p03 r004 on 0.03100373665931503
C000RG1 p03 r005 on 0.003894215022688671
C000RG1 p04 r000 on 0.036392430349582984
C000RG1 p04 r001 on 0.015925036124420106
C000RG1 p04 r002 on 0.037673985981189956
C000RG1 p04 r003 on 0.010249766775673687
C000RG1 p06 r000 on 0.04236632760412705
C000RG1 p06 r001 on 0.03938138768474155
C000RG1 p06 r002 on 0.03563715618424722
C000RG1 p06 r003 on 0.03652768334727609
C000RG1 p06 r004 on 0.03969206783785839
C000RG1 p06 r005 on 0.03838105115724897
C000RG1 p07 r002 on 0.0371570784850559
C000RG1 p07 r003 on 0.03851764329353309
C000RG1 p07 r004 on 0.03959564985930489
C000RG1 p07 r005 on 0.038741279438233585
C000RG1 p07 r006 on 0.014442609704160012
C000RG1 p07 r007 on 0.033747631632316775
C000RG1 p08 r000 on 0.03404358126093238
C000RG1 p08 r001 on 0.038518982432124114
C000RG1 p08 r002 on 0.037649881

(1.3993328706872517, 42)

In [16]:
tot_expo_det = 0
counts = 0
det = "C000RG1"

for period, runs in periods_dict.items():
    for run in runs:
        timestamp = meta2.datasets.runinfo[period][run]['phy']['start_key']
        chmap = meta.channelmap(timestamp)
        mass = chmap[det].production.mass_in_g
        if chmap[det].analysis.usability != 'on': 
            print(period, run)
            continue
        
        livetime_in_s = meta2.datasets.runinfo[period][run]["phy"]["livetime_in_s"]
        tmp = (mass
                / 1000
                * livetime_in_s
                / 60
                / 60
                / 24
                / 365
            )
        tot_expo_det+=tmp
        counts +=1
tot_expo_det, counts

KeyboardInterrupt: 